# Deep Learning Diagnostics Lab
작은 CIFAR-10 CNN을 SGD/AdamW로 학습하고 **학습동역학→표현→국소기하→loss surface** 순서로 진단합니다. 각 코드 셀 앞 설명에서 **왜 하는지/무엇을 읽는지**를 먼저 확인하세요. Drive에는 분석 결과만, 모델 `.pt`는 `/content/local_checkpoints`에만 저장합니다.

## 0. 준비
**목적:** 라이브러리, GPU, Drive 저장 폴더를 준비합니다. `FAST=True`는 Colab에서 짧게 돌리는 설정입니다.

In [ ]:
!pip -q install umap-learn tensorboard
import json,random,numpy as np,pandas as pd,matplotlib.pyplot as plt,torch,torch.nn as nn,torch.nn.functional as F
from pathlib import Path
from torch.utils.data import DataLoader,Subset
from torchvision import datasets,transforms
from torch.utils.tensorboard import SummaryWriter
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
import umap.umap_ as umap
from google.colab import drive
D=torch.device("cuda" if torch.cuda.is_available() else "cpu");SEED=7;FAST=True
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED);drive.mount("/content/drive")
R=Path("/content/drive/MyDrive/deep_learning_diagnostics");CSV=R/"csv";NPZ=R/"npz";FIG=R/"figures";TB=R/"tensorboard";SUM=R/"summaries";K=Path("/content/local_checkpoints")
for p in [CSV,NPZ,FIG,TB,SUM,K]:p.mkdir(parents=True,exist_ok=True)
print(torch.__version__,D)

## 1. 데이터/진단 시점
학습에는 crop/flip을 쓰지만 표현 비교는 **같은 이미지·순서**가 필요해 고정 eval 입력을 씁니다. representation은 매 epoch가 아니라 6 epoch 기준 `0,2,4,6`에서만 찍습니다.

In [ ]:
mean=(.4914,.4822,.4465);std=(.247,.2435,.2616)
tr=transforms.Compose([transforms.RandomCrop(32,padding=4),transforms.RandomHorizontalFlip(),transforms.ToTensor(),transforms.Normalize(mean,std)]);ev=transforms.Compose([transforms.ToTensor(),transforms.Normalize(mean,std)])
a=datasets.CIFAR10("/content/data",True,download=True,transform=tr);e=datasets.CIFAR10("/content/data",True,transform=ev)
p=torch.randperm(len(a),generator=torch.Generator().manual_seed(SEED)).tolist();nt,nv,E=(12000,2000,6) if FAST else (40000,5000,12);ti,vi=p[:nt],p[nt:nt+nv]
kw=dict(batch_size=256,num_workers=2,pin_memory=True);tl=DataLoader(Subset(a,ti),shuffle=True,**kw);tel=DataLoader(Subset(e,ti),**kw);vl=DataLoader(Subset(e,vi),**kw)
q=max(1,round(E/4));DE=sorted(set([0]+list(range(q,E+1,q))+[E]));print("diagnostic epochs",DE)

## 2. 모델
`stem→block1→block2→penultimate` 네 곳에서 representation을 꺼냅니다. convolution feature map은 공간 평균해 sample당 feature vector 하나로 바꿉니다.

In [ ]:
class Net(nn.Module):
 def __init__(s):
  super().__init__();s.stem=nn.Sequential(nn.Conv2d(3,32,3,padding=1),nn.ReLU());s.b1=nn.Sequential(nn.Conv2d(32,64,3,padding=1),nn.ReLU(),nn.MaxPool2d(2));s.b2=nn.Sequential(nn.Conv2d(64,128,3,padding=1),nn.ReLU(),nn.MaxPool2d(2));s.pool=nn.AdaptiveAvgPool2d(1);s.pen=nn.Linear(128,64);s.head=nn.Linear(64,10)
 def forward(s,x,features=False):
  f={};x=s.stem(x);f["stem"]=x.mean((2,3));x=s.b1(x);f["block1"]=x.mean((2,3));x=s.b2(x);f["block2"]=x.mean((2,3));x=s.pool(x).flatten(1);x=F.relu(s.pen(x));f["penultimate"]=x;o=s.head(x);return(o,f)if features else o
L=["stem","block1","block2","penultimate"];T={"stem":"stem.0.weight","block1":"b1.0.weight","block2":"b2.0.weight","penultimate":"pen.weight","head":"head.weight"};torch.manual_seed(SEED);INIT={k:v.cpu().clone()for k,v in Net().state_dict().items()}

## 3. 학습동역학
매 batch에 `gradient norm`과 `||ΔW||/||W||`를 기록합니다. **gradient=받은 학습신호**, **update/weight=실제로 움직인 비율**이라 둘을 함께 봐야 합니다.

In [ ]:
@torch.no_grad()
def evalm(m,l):
 m.eval();s=c=n=0
 for x,y in l:x,y=x.to(D),y.to(D);o=m(x);s+=F.cross_entropy(o,y,reduction="sum").item();c+=(o.argmax(1)==y).sum().item();n+=len(y)
 return s/n,c/n
def train(run,optname):
 m=Net().to(D);m.load_state_dict(INIT);opt=torch.optim.SGD(m.parameters(),lr=.08,momentum=.9)if optname=="sgd" else torch.optim.AdamW(m.parameters(),lr=2e-3);w=SummaryWriter(str(TB/run));H=[];Y=[];step=0;torch.save(m.state_dict(),K/f"{run}_e0.pt")
 for ep in range(1,E+1):
  m.train();s=c=n=0
  for x,y in tl:
   x,y=x.to(D),y.to(D);opt.zero_grad(set_to_none=True);o=m(x);loss=F.cross_entropy(o,y);loss.backward();P=dict(m.named_parameters());before={t:P[nm].detach().clone()for t,nm in T.items()};r={"epoch":ep,"step":step}
   for t,nm in T.items():r[t+"_grad"]=P[nm].grad.norm().item()
   opt.step();P=dict(m.named_parameters())
   for t,nm in T.items():d=(P[nm].detach()-before[t]).norm().item();r[t+"_update"]=d/(P[nm].detach().norm().item()+1e-12);w.add_scalar("grad/"+t,r[t+"_grad"],step);w.add_scalar("update/"+t,r[t+"_update"],step)
   Y.append(r);step+=1;s+=loss.item()*len(y);c+=(o.argmax(1)==y).sum().item();n+=len(y)
  vloss,vacc=evalm(m,vl);H.append({"epoch":ep,"train_loss":s/n,"train_acc":c/n,"val_loss":vloss,"val_acc":vacc});print(run,ep,vacc)
  if ep in DE:torch.save(m.state_dict(),K/f"{run}_e{ep}.pt")
 w.close();H=pd.DataFrame(H);Y=pd.DataFrame(Y);H.to_csv(CSV/f"{run}_history.csv",index=False);Y.to_csv(CSV/f"{run}_dynamics.csv",index=False);return m,H,Y
sgd,sh,sd=train("sgd","sgd");adam,ah,ad=train("adamw","adamw")

## 4. 먼저 기본 그래프
validation loss/accuracy로 문제를 위치화하고, layerwise gradient/update에서 **어느 층·시점이 이상한지** 찾습니다.

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(11,4))
for h,n in [(sh,"SGD"),(ah,"AdamW")]:ax[0].plot(h.epoch,h.val_loss,"o-",label=n);ax[1].plot(h.epoch,h.val_acc,"o-",label=n)
for a in ax:a.legend();plt.savefig(FIG/"training.png",dpi=160);plt.show()
def dyn(df,n):
 fig,ax=plt.subplots(1,2,figsize=(11,4))
 for l in T:ax[0].plot(df[l+"_grad"].rolling(20,min_periods=1).mean(),label=l);ax[1].plot(df[l+"_update"].rolling(20,min_periods=1).mean(),label=l)
 for a in ax:a.set_yscale("log");a.legend();plt.savefig(FIG/f"{n}_dynamics.png",dpi=160);plt.show()
dyn(sd,"sgd");dyn(ad,"adamw")

## 5. Representation snapshot
각 진단 시점에 **같은 validation 2,000장**의 네 층 activation을 NPZ로 저장합니다. 이후 rank/CKA/PCA/UMAP/local-PCA가 모두 이 데이터에서 출발합니다.

In [ ]:
@torch.no_grad()
def ext(m,maxn=2000):
 m.eval();Ff={k:[]for k in L};ys=[];ps=[];n=0
 for x,y in vl:
  o,f=m(x.to(D),True);take=min(len(y),maxn-n)
  for l in L:Ff[l].append(f[l][:take].cpu())
  ys.append(y[:take]);ps.append(o[:take].argmax(1).cpu());n+=take
  if n>=maxn:break
 return {k:torch.cat(v).numpy()for k,v in Ff.items()},torch.cat(ys).numpy(),torch.cat(ps).numpy()
def load(run,ep):return torch.load(K/f"{run}_e{ep}.pt",map_location="cpu")
def snaps(run):
 out={}
 for ep in DE:
  m=Net().to(D);m.load_state_dict(load(run,ep));f,y,p=ext(m);out[ep]=(f,y,p);np.savez_compressed(NPZ/f"{run}_e{ep}.npz",y=y,pred=p,**f)
 return out
S=snaps("sgd");A=snaps("adamw")

## 6. Effective rank + linear probe
**rank**는 표현이 몇 방향을 쓰는지, **probe**는 class 정보가 읽히는지 봅니다. `rank↓+probe↑`면 collapse보다 task-relevant compression 가능성이 있습니다.

In [ ]:
def erank(X):X=torch.tensor(X).float();X-=X.mean(0);s=torch.linalg.svdvals(X);p=s.square();p/=p.sum();return torch.exp(-(p*torch.log(p.clamp_min(1e-12))).sum()).item()
rows=[]
for run,Z in [("sgd",S),("adamw",A)]:
 for ep in DE:
  for l in L:rows.append([run,ep,l,erank(Z[ep][0][l])])
rd=pd.DataFrame(rows,columns=["run","epoch","layer","rank"]);rd.to_csv(CSV/"rank.csv",index=False)
for run in ["sgd","adamw"]:
 for l in L:q=rd[(rd.run==run)&(rd.layer==l)];plt.plot(q.epoch,q["rank"],"o-",label=l)
 plt.legend();plt.title(run+" effective rank");plt.savefig(FIG/f"{run}_rank.png",dpi=160);plt.show()
trf,try_,_=ext(sgd,5000);sf,yv,pv=S[E]
def probe(X,y,V,z):
 sc=StandardScaler();X=torch.tensor(sc.fit_transform(X).astype("float32"),device=D);V=torch.tensor(sc.transform(V).astype("float32"),device=D);y=torch.tensor(y,device=D);z=torch.tensor(z,device=D);h=nn.Linear(X.shape[1],10).to(D);o=torch.optim.AdamW(h.parameters(),lr=.08)
 for _ in range(120):o.zero_grad();loss=F.cross_entropy(h(X),y);loss.backward();o.step()
 return(h(V).argmax(1)==z).float().mean().item()
pd.DataFrame([[l,probe(trf[l],try_,sf[l],yv)]for l in L],columns=["layer","probe_accuracy"]).to_csv(CSV/"probe.csv",index=False)

## 7. CKA
`CKA to init`가 내려가면 초기 표현에서 멀어지고, `CKA to final`이 올라가면 최종 표현이 형성되고 있습니다. **어느 층이 언제 바뀌는지**를 봅니다.

In [ ]:
def cka(X,Y):X=torch.tensor(X).float();Y=torch.tensor(Y).float();X-=X.mean(0);Y-=Y.mean(0);return((X.T@Y).square().sum()/(((X.T@X).square().sum().sqrt()*(Y.T@Y).square().sum().sqrt())+1e-12)).item()
rows=[]
for run,Z in [("sgd",S),("adamw",A)]:
 for ep in DE:
  for l in L:rows.append([run,ep,l,cka(Z[ep][0][l],Z[0][0][l]),cka(Z[ep][0][l],Z[E][0][l])])
cd=pd.DataFrame(rows,columns=["run","epoch","layer","to_init","to_final"]);cd.to_csv(CSV/"cka.csv",index=False)
for run in ["sgd","adamw"]:
 fig,ax=plt.subplots(1,2,figsize=(11,4))
 for l in L:q=cd[(cd.run==run)&(cd.layer==l)];ax[0].plot(q.epoch,q.to_init,"o-",label=l);ax[1].plot(q.epoch,q.to_final,"o-",label=l)
 for a in ax:a.set_ylim(0,1.02);a.legend();plt.savefig(FIG/f"{run}_cka.png",dpi=160);plt.show()

## 8. PCA / UMAP
PCA는 전역 큰 분산, UMAP은 local neighborhood를 2D로 봅니다. **검은 테두리=오분류**라 class 경계와 실패 위치가 같이 보입니다.

In [ ]:
for run,Z in [("sgd",S),("adamw",A)]:
 for ep in DE:
  X=StandardScaler().fit_transform(Z[ep][0]["penultimate"]);y=Z[ep][1];p=Z[ep][2]
  for name,r in [("pca",PCA(2)),("umap",umap.UMAP(n_components=2,n_neighbors=20,min_dist=.15,random_state=SEED))]:
   Q=r.fit_transform(X);ok=y==p;plt.scatter(Q[:,0],Q[:,1],c=y,cmap="tab10",s=9,alpha=.6);plt.scatter(Q[~ok,0],Q[~ok,1],facecolors="none",edgecolors="black",s=30);plt.title(f"{run} {name} e{ep}");plt.savefig(FIG/f"{run}_{name}_e{ep}.png",dpi=160);plt.show()

## 9. Local PCA
각 sample의 가까운 이웃 30개에 PCA를 하고 **90% 분산을 설명하는 차원 수**를 local complexity proxy로 씁니다. 오분류 주변이 더 복잡한지 비교합니다.

In [ ]:
X=StandardScaler().fit_transform(sf["penultimate"]);nnn=NearestNeighbors(n_neighbors=31).fit(X);_,ix=nnn.kneighbors(X);ld=[]
for i in range(len(X)):Z=X[ix[i,1:]];Z-=Z.mean(0);s=np.linalg.svd(Z,compute_uv=False);q=s*s;ld.append(np.searchsorted(np.cumsum(q)/q.sum(),.9)+1)
ld=np.array(ld);ok=yv==pv;pd.DataFrame({"label":yv,"pred":pv,"correct":ok,"local_dim90":ld}).to_csv(CSV/"local_dim.csv",index=False);plt.boxplot([ld[ok],ld[~ok]],tick_labels=["correct","wrong"]);plt.savefig(FIG/"local_dim.png",dpi=160);plt.show()

## 10. Hessian top eigenvalue
전체 Hessian 대신 Hessian-vector product로 top eigenvalue를 추정합니다. 값이 크면 현재 parameter 주변에 loss가 급하게 휘는 방향이 있다는 뜻입니다.

In [ ]:
def htop(st,it=6):
 m=Net().to(D);m.load_state_dict(st);x,y=next(iter(tel));x,y=x[:128].to(D),y[:128].to(D);ps=list(m.parameters());v=[torch.randn_like(p)for p in ps]
 for _ in range(it):n=torch.sqrt(sum((a*a).sum()for a in v));v=[a/n for a in v];g=torch.autograd.grad(F.cross_entropy(m(x),y),ps,create_graph=True);hv=torch.autograd.grad(sum((a*b).sum()for a,b in zip(g,v)),ps);eig=sum((a*b).sum()for a,b in zip(v,hv)).item();v=[a.detach()for a in hv]
 return eig
hd=pd.DataFrame([["SGD init",htop(load("sgd",0))],["SGD final",htop(load("sgd",E))],["AdamW final",htop(load("adamw",E))]],columns=["condition","lambda_max"]);hd.to_csv(CSV/"hessian.csv",index=False);plt.bar(hd.condition,hd.lambda_max);plt.xticks(rotation=15);plt.savefig(FIG/"hessian.png",dpi=160);plt.show()

## 11. Weight interpolation
두 weight를 `alpha=0→1`로 선형 혼합합니다. 중간에서 loss가 솟으면 두 solution이 **직선 low-loss path로 연결되지 않음**을 뜻합니다.

In [ ]:
def mix(a,b,t):return{k:((1-t)*a[k]+t*b[k]if torch.is_floating_point(a[k])else(a[k]if t<.5 else b[k]))for k in a}
@torch.no_grad()
def curve(a,b,name):
 m=Net().to(D);r=[]
 for t in np.linspace(0,1,21):m.load_state_dict(mix(a,b,float(t)));L,A=evalm(m,vl);r.append([name,t,L,A])
 return r
mid=DE[-2];ID=pd.DataFrame(curve(load("sgd",mid),load("sgd",E),"same-run")+curve(load("sgd",E),load("adamw",E),"cross-optimizer"),columns=["path","alpha","loss","accuracy"]);ID.to_csv(CSV/"interpolation.csv",index=False)
for n,q in ID.groupby("path"):plt.plot(q.alpha,q.loss,"o-",label=n)
plt.legend();plt.savefig(FIG/"interpolation.png",dpi=160);plt.show()

## 12. TensorBoard / 저장 확인
TensorBoard에서 batch별 gradient/update를 보고, 마지막 assert로 Drive에 모델 `.pt`가 없는지 확인합니다.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/deep_learning_diagnostics/tensorboard

In [ ]:
assert not any(R.rglob("*.pt")),"model weight found on Drive"
print("OK: analysis on Drive, model weights local only")

## 결과 읽기
`loss/accuracy → gradient/update → rank+probe → CKA → PCA/UMAP → local PCA → Hessian → interpolation` 순서로 보세요. 예: validation 정체 + block2 update 작음 + rank 변화 없음 + CKA-to-init 높음 → **block2가 feature learning을 거의 안 하는가?**라는 가설을 세우고, 다음 실험에서 LR/normalization/width/optimizer 중 하나만 바꿔 재검증합니다.